# 00 · Evaluate the source (StatsBomb)

First contact with an unfamiliar source: what entities exist, how big are they, how nested,
how often do they change, and what quirks will bite downstream? Uses **statsbombpy** (as suggested
in the brief) for quick exploration – production ingestion lands raw JSON instead (see ADR-001).

In [ ]:
%pip install -q statsbombpy

In [ ]:
import warnings, collections
import pandas as pd
from statsbombpy import sb
warnings.filterwarnings("ignore")   # "credentials were not supplied. open data access only"

comps = sb.competitions()
display(comps[["competition_id", "season_id", "competition_name", "season_name", "match_updated",
               "match_available_360"]].sort_values("match_updated", ascending=False).head(20))

In [ ]:
matches = sb.matches(competition_id=55, season_id=282)   # UEFA Euro 2024
print(len(matches), "matches;", matches["last_updated"].min(), "->", matches["last_updated"].max())
display(matches[["match_id", "match_date", "home_team", "away_team", "home_score", "away_score",
                 "competition_stage", "last_updated", "last_updated_360"]].head())

In [ ]:
events = sb.events(match_id=3943043)   # the final
print(events.shape, "rows x cols  <- statsbombpy flattens nested JSON into ~100+ sparse columns")
display(events["type"].value_counts().head(15))
print("share of nulls per column (top 15):")
display(events.isna().mean().sort_values(ascending=False).head(15))

### Findings that shaped the design
* **Optional keys are omitted**, not null – a pass with no `outcome` is *complete*; absent flags mean `False`.
* **Variable shapes** – `shot.end_location` is 2-D or 3-D; freeze-frames & tactics are nested arrays → child tables.
* **Data is revised long after the match** – Euro 2024 `last_updated` values are from 2026 → watermark-based incremental loads.
* **Clock quirks** – timestamps restart each period; extra-time lineup stints are chained in string order (bug) → minutes rebuilt.
* **No DOB / external IDs in open-data lineups** → identity resolution against Catapult / Transfermarkt is required.
* **statsbombpy swallows HTTP errors** (returns `[]`) → production client raises on non-200s.
* **Coordinates**: 120×80 units, each team attacking left→right; tracking providers use metres from centre → conversion macros.